# Notebook 1: Simulating Heteroscedasticity & Its Impact on OLS
**Prepared for:** Assumptions of Least Square Regressions Lecture Suite  
**Author:** Nick Lim

In this notebook, you will get hands-on experience simulating data where the assumption of **homoscedasticity** (constant error variance across all observations) is violated. We will explore:
1. How heteroscedasticity affects OLS coefficient estimates (`beta`).
2. Why standard OLS confidence intervals become unreliable under heteroscedasticity.
3. How to diagnose heteroscedasticity using residuals vs. fitted plots and fix it using robust standard errors (`HC3`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Polyfill for matplotlib 3.9+ compatibility with matplotlib-inline
if not hasattr(matplotlib.rcParams, '_get'):
    matplotlib.rcParams._get = lambda key: matplotlib.rcParams[key]

# Set random seed for exact reproducibility
np.random.seed(42)
sns.set_theme(style="whitegrid")
plt.rcParams["font.size"] = 12

## 1. Simulating Homoscedastic vs. Heteroscedastic Data

Let's create two datasets from the exact same true underlying model:
$$Y_i = 3.0 + 2.0 X_i + \epsilon_i$$

- **Homoscedastic Dataset:** $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$ where $\sigma = 1.5$ (constant across all $X$).
- **Heteroscedastic Dataset:** $\epsilon_i \sim \mathcal{N}(0, \sigma_i^2)$ where $\sigma_i = 0.4 \cdot X_i$ (variance grows like a megaphone as $X$ increases).

In [ ]:
n = 200
x = np.random.uniform(1, 10, n)

# 1. Homoscedastic error
err_hom = np.random.normal(0, 1.5, n)
y_hom = 3.0 + 2.0 * x + err_hom

# 2. Heteroscedastic error (standard deviation scales linearly with x)
err_het = np.random.normal(0, 0.4 * x)
y_het = 3.0 + 2.0 * x + err_het

# Plotting both side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Homoscedastic scatter
axes[0].scatter(x, y_hom, color="#2563eb", alpha=0.7)
axes[0].plot(x, 3.0 + 2.0 * x, color="#1e293b", linewidth=2, linestyle="--", label="True Line ($Y = 3 + 2X$)")
axes[0].set_title("Homoscedastic Data (Constant Error Spread)")
axes[0].set_xlabel("Predictor ($X$)")
axes[0].set_ylabel("Outcome ($Y$)")
axes[0].legend()

# Heteroscedastic scatter
axes[1].scatter(x, y_het, color="#dc2626", alpha=0.7)
axes[1].plot(x, 3.0 + 2.0 * x, color="#1e293b", linewidth=2, linestyle="--", label="True Line ($Y = 3 + 2X$)")
axes[1].set_title("Heteroscedastic Data (Megaphone Error Spread)")
axes[1].set_xlabel("Predictor ($X$)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 2. Fitting OLS and Diagnostic Plots (`Residuals vs. Fitted`)

Let's see what happens when we fit an Ordinary Least Squares (`sm.OLS`) model to both datasets and plot the **Residuals against Fitted Values (`Y_hat`)**.

In [ ]:
X_matrix = sm.add_constant(x)

mod_hom = sm.OLS(y_hom, X_matrix).fit()
mod_het = sm.OLS(y_het, X_matrix).fit()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Homoscedastic Residuals
axes[0].scatter(mod_hom.fittedvalues, mod_hom.resid, color="#2563eb", alpha=0.7)
axes[0].axhline(0, color="#1e293b", linestyle="--", linewidth=2)
axes[0].set_title("Homoscedastic Residuals vs Fitted")
axes[0].set_xlabel("Fitted Values ($\\hat{Y}$)")
axes[0].set_ylabel("Residuals ($e_i$)")

# Heteroscedastic Residuals
axes[1].scatter(mod_het.fittedvalues, mod_het.resid, color="#dc2626", alpha=0.7)
axes[1].axhline(0, color="#1e293b", linestyle="--", linewidth=2)
axes[1].set_title("Heteroscedastic Residuals vs Fitted ('Megaphone')")
axes[1].set_xlabel("Fitted Values ($\\hat{Y}$)")
axes[1].set_ylabel("Residuals ($e_i$)")

plt.tight_layout()
plt.show()

## 3. Monte Carlo Simulation: Does Heteroscedasticity Bias $\hat{\beta}$?

Let's run 1,000 simulations where we generate heteroscedastic data and calculate the OLS slope $\hat{\beta}_1$ each time. Is the average of our estimated slopes still equal to the true slope of `2.0`?

In [ ]:
n_sims = 1000
slopes_hom = []
slopes_het = []

for _ in range(n_sims):
    x_sim = np.random.uniform(1, 10, n)
    # Hom
    y_h = 3.0 + 2.0 * x_sim + np.random.normal(0, 1.5, n)
    slopes_hom.append(sm.OLS(y_h, sm.add_constant(x_sim)).fit().params[1])
    # Het
    y_ht = 3.0 + 2.0 * x_sim + np.random.normal(0, 0.4 * x_sim, n)
    slopes_het.append(sm.OLS(y_ht, sm.add_constant(x_sim)).fit().params[1])

print(f"Mean Homoscedastic Slope: {np.mean(slopes_hom):.4f} (True = 2.0)")
print(f"Mean Heteroscedastic Slope: {np.mean(slopes_het):.4f} (True = 2.0)")

plt.figure(figsize=(9, 5))
sns.kdeplot(slopes_hom, color="#2563eb", fill=True, label="Homoscedastic Slopes")
sns.kdeplot(slopes_het, color="#dc2626", fill=True, alpha=0.3, label="Heteroscedastic Slopes")
plt.axvline(2.0, color="#1e293b", linestyle="--", linewidth=2, label="True Slope ($\\beta_1 = 2.0$)")
plt.title("Distribution of Estimated Slopes Over 1,000 Simulations")
plt.xlabel("Estimated Slope ($\\hat{\\beta}_1$)")
plt.legend()
plt.show()

### Key Takeaway:
- **OLS estimates remain UNBIASED under heteroscedasticity!** Both curves center directly on `2.0`.
- However, notice that the spread (variance) of the slopes under heteroscedasticity is wider and standard OLS standard errors underestimate or distort our uncertainty. Let's see how to fix this using **Robust Standard Errors (`HC3`)**.

In [ ]:
# Standard OLS summary (assuming homoscedasticity)
print("=== Standard OLS Summary (Vulnerable to Heteroscedasticity) ===")
print(mod_het.summary().tables[1])

# Robust OLS summary using White's HC3 standard errors
mod_het_robust = mod_het.get_robustcov_results(cov_type="HC3")
print("\n=== Heteroscedasticity-Robust Summary (HC3 Standard Errors) ===")
print(mod_het_robust.summary().tables[1])